In [ ]:
# P0.3 (replikasi): pin stack era 4.x — transformers tidak dipakai family Keras,
# tapi pip pin tetap untuk konsistensi environment jika sel import menyentuh HF.
!pip uninstall -y torchao
!pip install --force-reinstall --no-deps "transformers==4.46.3" "peft==0.13.2" "tokenizers==0.20.3" "huggingface-hub==0.26.5"


In [ ]:
# P0.1 (replikasi): paksa 1 GPU.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))

import sys
import tensorflow as tf
print("python :", sys.version)
print("tf     :", tf.__version__)
print("gpus   :", tf.config.list_physical_devices("GPU"))


In [ ]:
from __future__ import annotations
# =====================================================
# SUMBER KEBENARAN: src/ (disuntik oleh tools/generate_notebook.py)
# Jangan edit langsung di notebook - edit src/ lalu generate ulang.
# =====================================================

# --- src/config.py ---
"""Konfigurasi eksperimen: load dari YAML/JSON dan bantu membenamkan dict ke notebook.

Sumber kebenaran konfigurasi = file di `configs/`. Generator membaca file ini,
lalu membenamkan representasi literal dict-nya ke sel Config notebook (sel 6),
sehingga notebook Kaggle tidak butuh PyYAML.
"""

import json
from pathlib import Path
from typing import Any


def load_config(path: str | Path) -> dict[str, Any]:
    """Muat file config (.yaml/.yml/.json) menjadi dict."""
    p = Path(path)
    text = p.read_text(encoding="utf-8").strip()
    if p.suffix.lower() in (".yaml", ".yml"):
        try:
            import yaml
        except ImportError as e:
            raise ImportError(
                "PyYAML dibutuhkan untuk config YAML: pip install pyyaml"
            ) from e
        cfg = yaml.safe_load(text)
        if not isinstance(cfg, dict):
            raise ValueError(f"Config {p} harus berupa mapping YAML, bukan {type(cfg)}")
        return cfg
    return json.loads(text)


def config_repr(cfg: dict[str, Any]) -> str:
    """Representasi Python literal (json.dumps) untuk dibenamkan di sel notebook."""
    return json.dumps(cfg, indent=4, ensure_ascii=False)


def config_snippet(cfg: dict[str, Any], var_name: str = "CONFIG") -> str:
    """Source untuk sel Config: `CONFIG = {...}`."""
    return f"{var_name} = {config_repr(cfg)}"

# --- src/keras_data.py ---
"""Data untuk LSTM/BiLSTM (Keras): loader fleksibel + tokenizer + padding.

Sumber kebenaran loading data untuk eksperimen Keras (family keras_lstm /
keras_bilstm). Sel notebook menyuntik source file ini (via generator) sehingga
self-contained di Kaggle. Data diambil dari kolom `clean_text_lstm` di CSV
dataset (label corrected), split 80:20 lalu 90:10 (protokol sama dgn eksperimen HF).
"""

import os
from typing import Any

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

COL_TEXT = "clean_text_lstm"
COL_LABEL = "label"
CSV_NAME = "data_preprocessed_with_emoticon.csv"


def find_dataset_csv() -> str:
    """Cari CSV dataset di /kaggle/input (path mount CLI 2.x vs lama)."""
    mounted = []
    for root, _dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f == CSV_NAME:
                mounted.append(os.path.join(root, f))
    if not mounted:
        raise FileNotFoundError(
            f"Dataset '{CSV_NAME}' tidak ditemukan di /kaggle/input. "
            "Cek dataset_sources di kernel-metadata.json."
        )
    return mounted[0]


def load_lstm_data(
    max_words: int = 10000,
    max_len: int = 50,
    test_size: float = 0.2,
    val_size: float = 0.1,
    random_state: int = 42,
) -> dict[str, Any]:
    """Load CSV, split stratify 80:20 -> 90:10, tokenizer, pad_sequences.

    Mengembalikan dict: X_train_pad, X_val_pad, X_test_pad, y_train, y_val, y_test.
    """
    path = find_dataset_csv()
    print("CSV ditemukan di:", path)
    df = pd.read_csv(path)
    if COL_TEXT not in df.columns:
        raise ValueError(
            f"Kolom '{COL_TEXT}' tidak ditemukan di CSV. Kolom tersedia: {df.columns.tolist()}"
        )
    if COL_LABEL not in df.columns:
        raise ValueError(f"Kolom '{COL_LABEL}' tidak ditemukan di CSV.")

    df[COL_TEXT] = df[COL_TEXT].fillna("").astype(str)

    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df[COL_LABEL]
    )
    X_train, y_train = train_df[COL_TEXT].values, train_df[COL_LABEL].values
    X_test, y_test = test_df[COL_TEXT].values, test_df[COL_LABEL].values

    X_train_final, X_val, y_train_final, y_val = train_test_split(
        X_train, y_train, test_size=val_size, stratify=y_train, random_state=random_state
    )

    tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_train_final)

    X_train_pad = pad_sequences(
        tokenizer.texts_to_sequences(X_train_final), maxlen=max_len, padding="post"
    )
    X_val_pad = pad_sequences(
        tokenizer.texts_to_sequences(X_val), maxlen=max_len, padding="post"
    )
    X_test_pad = pad_sequences(
        tokenizer.texts_to_sequences(X_test), maxlen=max_len, padding="post"
    )

    print(f"Shape train: {X_train_pad.shape} | val: {X_val_pad.shape} | test: {X_test_pad.shape}")
    print("Distribusi train final:", pd.Series(y_train_final).value_counts().sort_index().to_dict())
    print("Distribusi val        :", pd.Series(y_val).value_counts().sort_index().to_dict())
    print("Distribusi test       :", pd.Series(y_test).value_counts().sort_index().to_dict())

    return {
        "X_train_pad": X_train_pad,
        "X_val_pad": X_val_pad,
        "X_test_pad": X_test_pad,
        "y_train": y_train_final,
        "y_val": y_val,
        "y_test": y_test,
    }

# --- src/keras_model.py ---
"""Model LSTM/BiLSTM (Keras): builder tunggal untuk family keras_lstm/keras_bilstm.

- LSTM: Embedding -> LSTM(units) -> Dropout -> Dense(64, relu) -> Dense(3, softmax)
- BiLSTM: sama, LSTM dibungkus Bidirectional
Sumber kebenaran: file ini disuntik ke sel notebook oleh generator.
"""

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Bidirectional, Dense, Dropout, Embedding, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam


def build_lstm_model(
    max_words: int = 10000,
    max_len: int = 50,
    embedding_dim: int = 128,
    units: int = 64,
    dropout: float = 0.3,
    learning_rate: float = 1e-3,
    bidirectional: bool = False,
) -> Sequential:
    """Bangun model Keras (LSTM atau BiLSTM) untuk klasifikasi 3 kelas."""
    model = Sequential()
    model.add(
        Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len)
    )
    if bidirectional:
        model.add(Bidirectional(LSTM(units)))
    else:
        model.add(LSTM(units))
    model.add(Dropout(dropout))
    model.add(Dense(64, activation="relu"))
    model.add(Dense(3, activation="softmax"))

    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=Adam(learning_rate=learning_rate),
        metrics=["accuracy"],
    )
    return model

# --- src/metrics.py ---
"""Metrik evaluasi: compute_metrics (HF) + softmax numpy (untuk simpan probabilitas)."""

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

LABEL_NAMES = ["negatif", "netral", "positif"]


def compute_metrics(eval_pred):
    """Metrik untuk HF Trainer (average='macro', zero_division=0)."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }


def softmax_np(logits: np.ndarray) -> np.ndarray:
    """Softmax stabil (numerik) di axis=1."""
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def prediction_frame(
    texts,
    y_true,
    logits: np.ndarray,
    prob_cols=("prob_negatif", "prob_netral", "prob_positif"),
):
    """DataFrame per-sampel: teks, label aktual/prediksi, probabilitas per kelas."""
    import pandas as pd

    y_pred = np.argmax(logits, axis=1)
    P = softmax_np(logits)
    return pd.DataFrame(
        {
            "text": pd.Series(texts),
            "label_aktual": pd.Series(y_true),
            "label_prediksi": pd.Series(y_pred),
            prob_cols[0]: P[:, 0],
            prob_cols[1]: P[:, 1],
            prob_cols[2]: P[:, 2],
        }
    )

# --- src/summary.py ---
"""Auto Experiment Summary: menulis metadata run (config, commit, dataset MD5) ke file & stdout."""

import hashlib
import json
import subprocess
from pathlib import Path
from typing import Any


def git_commit_short() -> str:
    """Hash commit git saat ini (atau 'unknown' bila bukan repo git)."""
    try:
        out = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            capture_output=True,
            text=True,
            timeout=10,
        )
        return out.stdout.strip() or "unknown"
    except Exception:
        return "unknown"


def file_md5(path: str | Path) -> str:
    """MD5 file (untuk audit data lineage)."""
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def experiment_summary(
    exp_id: str,
    config: dict[str, Any],
    metrics: dict[str, Any],
    csv_path: str | None = None,
    dataset_path: str | None = None,
    out_path: str | None = None,
) -> dict[str, Any]:
    """Buat dict ringkasan + tulis file JSON (opsional) + cetak ke stdout.

    Metrics contoh: {"accuracy": ..., "f1_macro": ..., "netral_f1": ...}
    """
    summary = {
        "experiment": exp_id,
        "commit": git_commit_short(),
        "config": config,
        "metrics": metrics,
    }
    if csv_path:
        summary["prediction_csv"] = csv_path
    if dataset_path:
        summary["dataset_md5"] = file_md5(dataset_path)

    if out_path:
        Path(out_path).parent.mkdir(parents=True, exist_ok=True)
        Path(out_path).write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

    print("=" * 60)
    print("AUTO EXPERIMENT SUMMARY")
    print("=" * 60)
    print(f"Experiment : {exp_id}")
    print(f"Commit     : {summary.get('commit')}")
    if dataset_path:
        print(f"Dataset MD5: {summary['dataset_md5']}")
    print(f"Config     : {json.dumps(config, ensure_ascii=False)}")
    print(f"Metrics    : {json.dumps(metrics, ensure_ascii=False)}")
    if out_path:
        print(f"Summary    : {out_path}")
    return summary

In [ ]:
# =====================================================
# SET SEED
# =====================================================
import random
seed = 42
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)
print("GPU tersedia:", tf.config.list_physical_devices("GPU") != [])


In [ ]:
# =====================================================
# DATASET (loader fleksibel + kolom clean_text_lstm)
# =====================================================
d = load_lstm_data(
    max_words=CONFIG["params"]["max_words"],
    max_len=CONFIG["params"]["max_len"],
    test_size=0.2,
    val_size=0.1,
    random_state=42,
)


In [ ]:
# =====================================================
# CONFIG (dibenamkan dari configs/exp_bilstm_v2.yaml)
# =====================================================
CONFIG = {
    "exp_id": "exp_bilstm_v2",
    "family": "keras_bilstm",
    "title": "Thesis BiLSTM v2",
    "description": "B3 - Re-run BiLSTM empiris label corrected (lr 1e-3, BiLSTM 64, EarlyStopping patience 3) + class weight",
    "params": {
        "max_words": 10000,
        "max_len": 50,
        "embedding_dim": 128,
        "epochs": 20
    },
    "variants": [
        {
            "name": "empiris_lr1e3",
            "learning_rate": 0.001,
            "units": 64,
            "dropout": 0.3,
            "batch_size": 32,
            "patience": 5,
            "restore_best_weights": true,
            "reduce_lr": true,
            "class_weight": false
        },
        {
            "name": "class_weight_lr1e3",
            "learning_rate": 0.001,
            "units": 64,
            "dropout": 0.3,
            "batch_size": 32,
            "patience": 5,
            "restore_best_weights": true,
            "reduce_lr": true,
            "class_weight": true
        }
    ],
    "dataset_sources": [
        "emanuelembuaijdak/thesis-indobert-processed-data"
    ]
}
print(json.dumps(CONFIG, indent=2))

In [ ]:
# =====================================================
# MODEL FACTORY (Keras, dari CONFIG)
# =====================================================
def make_model(variant):
    return build_lstm_model(
        max_words=CONFIG["params"]["max_words"],
        max_len=CONFIG["params"]["max_len"],
        embedding_dim=CONFIG["params"].get("embedding_dim", 128),
        units=variant["units"],
        dropout=variant["dropout"],
        learning_rate=variant["learning_rate"],
        bidirectional=(CONFIG["family"] == "keras_bilstm"),
    )


In [ ]:
# =====================================================
# TRAINING (loop variants + sanity collapse)
# =====================================================
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

variants = CONFIG["variants"]
val_results = []
best_model = None
best_f1 = -1
best_variant = None

for v in variants:
    name = v["name"]
    print("\n" + "=" * 60)
    print("VARIANT:", name, "|", v)
    print("=" * 60)

    K.clear_session()
    tf.random.set_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = make_model(v)

    callbacks = [EarlyStopping(
        monitor="val_loss",
        patience=v.get("patience", 5),
        restore_best_weights=v.get("restore_best_weights", True),
    )]
    if v.get("reduce_lr"):
        callbacks.append(ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6, verbose=1,
        ))

    class_weight = None
    if v.get("class_weight"):
        classes = np.unique(d["y_train"])
        cw_arr = compute_class_weight("balanced", classes=classes, y=d["y_train"])
        class_weight = dict(enumerate(cw_arr))
        print("Class weights:", class_weight)

    history = model.fit(
        d["X_train_pad"],
        d["y_train"],
        validation_data=(d["X_val_pad"], d["y_val"]),
        epochs=CONFIG["params"]["epochs"],
        batch_size=v.get("batch_size", 32),
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=1,
    )

    y_val_pred = np.argmax(model.predict(d["X_val_pad"], verbose=0), axis=1)
    acc_val = accuracy_score(d["y_val"], y_val_pred)
    _, _, f1_macro, _ = precision_recall_fscore_support(
        d["y_val"], y_val_pred, average="macro", zero_division=0
    )

    # --- Sanity check (P0): deteksi collapse ---
    maj = pd.Series(d["y_val"]).mode()[0]
    p_maj = float((d["y_val"] == maj).mean())
    f1_maj = (2 * p_maj / (1 + p_maj)) / 3
    print("Distribusi prediksi val :", pd.Series(y_val_pred).value_counts().sort_index().to_dict())
    print("Baseline mayoritas val  : acc=" + str(round(p_maj, 4)) + " macro_f1=" + str(round(f1_maj, 4)))
    status = "COLLAPSE" if f1_macro <= f1_maj + 1e-6 else "OK"
    print("Val accuracy:", round(acc_val, 4), "| Val macro F1:", round(f1_macro, 4), "| STATUS:", status)

    val_results.append({
        "name": name,
        "learning_rate": v["learning_rate"],
        "units": v["units"],
        "dropout": v["dropout"],
        "batch_size": v.get("batch_size", 32),
        "class_weight": v.get("class_weight", False),
        "accuracy_val": acc_val,
        "f1_macro_val": f1_macro,
        "status": status,
    })

    if f1_macro > best_f1:
        best_f1 = f1_macro
        best_model = model
        best_variant = name

df_val = pd.DataFrame(val_results).sort_values("f1_macro_val", ascending=False)
print("\n=== RINGKASAN VALIDATION ===")
print(df_val.to_string(index=False))
df_val.to_csv("exp_bilstm_v2_val.csv", index=False)
print("Best variant (val macro F1):", best_variant, "| F1:", round(best_f1, 4))


In [ ]:
# =====================================================
# EVALUASI TEST (best model) + SIMPAN PROBABILITAS
# =====================================================
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

P = best_model.predict(d["X_test_pad"], verbose=0)  # Keras softmax -> proba per kelas
y_pred_test = np.argmax(P, axis=1)

print(classification_report(d["y_test"], y_pred_test, target_names=LABEL_NAMES, zero_division=0))
print("Distribusi prediksi:", pd.Series(y_pred_test).value_counts().sort_index().to_dict())
print("Distribusi aktual  :", pd.Series(d["y_test"]).value_counts().sort_index().to_dict())

hasil = pd.DataFrame({
    "label_aktual": pd.Series(d["y_test"]),
    "label_prediksi": pd.Series(y_pred_test),
    "prob_negatif": P[:, 0],
    "prob_netral": P[:, 1],
    "prob_positif": P[:, 2],
})
fname = "exp_bilstm_v2_test.csv"
hasil.to_csv(fname, index=False)
print("Tersimpan:", fname)

precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    d["y_test"], y_pred_test, average="macro", zero_division=0
)
accuracy = accuracy_score(d["y_test"], y_pred_test)

cm = confusion_matrix(d["y_test"], y_pred_test)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.title("Confusion Matrix - exp_bilstm_v2")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
# =====================================================
# SAVE ARTIFACT + AUTO EXPERIMENT SUMMARY
# =====================================================
import json

metrics = {
    "accuracy": accuracy,
    "precision_macro": precision_macro,
    "recall_macro": recall_macro,
    "f1_macro": f1_macro,
}
summary = experiment_summary(
    exp_id='exp_bilstm_v2',
    config=CONFIG,
    metrics=metrics,
    csv_path="exp_bilstm_v2_test.csv",
    out_path="exp_bilstm_v2_summary.json",
)
